# Fase 2: Preprocesamiento y Transformación de Logs de Seguridad
**Proyecto:** Análisis de datos de amenazas de ciberseguridad  
**Integrantes:** Jennifer Nilo – Patricio Núñez  

## 1. Contexto y Objetivos
En la operación de un Centro de Operaciones de Seguridad (SOC), la calidad de los datos ingeridos es fundamental. Los logs con valores nulos o formatos incorrectos generan falsos positivos y aumentan la fatiga de alertas. 

El objetivo de esta Fase 2 es aplicar un pipeline secuencial para depurar nuestro dataset de 6 millones de registros mediante:
1. **Exploración e Ingesta:** Carga segura de datos.
2. **Limpieza:** Tratamiento de valores nulos (NA).
3. **Transformación:** Conversión de tipos de datos (*Casting*).
4. **Normalización:** Escalamiento de variables numéricas.
5. **Validación:** Comprobación de la integridad del dataset final.

In [1]:
# Importación de librerías base
import pandas as pd
import numpy as np

# Fijar la semilla de aleatoriedad para garantizar la reproducibilidad del experimento
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Librerías importadas. Entorno de la Fase 2 inicializado.")

Librerías importadas. Entorno de la Fase 2 inicializado.


## 2. Obtención e Ingesta de Datos
Para procesar el volumen masivo de información, estructuramos la carga mediante una función modular. Utilizamos el parámetro `low_memory=False` en Pandas para evitar advertencias por tipos de datos mixtos durante la lectura inicial del archivo CSV crudo.

In [2]:
def cargar_datos(ruta):
    """Función para cargar el CSV de forma segura y estructurada."""
    print(f"Cargando datos desde: {ruta}...")
    dataframe = pd.read_csv(ruta, low_memory=False)
    print(f"¡Carga exitosa! Filas: {dataframe.shape[0]:,}, Columnas: {dataframe.shape[1]}")
    return dataframe

# La funcion se ejecuta con los datos almacenados en /data/raw
ruta_archivo = '../data/raw/cybersecurity_threat_detection_logs.csv'
df_crudo = cargar_datos(ruta_archivo)

Cargando datos desde: ../data/raw/cybersecurity_threat_detection_logs.csv...
¡Carga exitosa! Filas: 6,000,000, Columnas: 10


## 3. Limpieza Exhaustiva (Manejo de Valores NA)
La depuración de valores nulos se divide en dos criterios operativos:
* **Descarte de registros inválidos:** Si un log no contiene dirección IP o Timestamp, pierde su valor analítico y es eliminado.
* **Imputación de metadatos:** Si faltan datos complementarios (como el `user_agent`), se rellenan con un valor por defecto ("Desconocido") para no perder el resto de la información útil de esa fila.

In [2]:
def limpiar_nulos(df):
    """Elimina filas sin IP o Timestamp, y rellena los vacíos en otras columnas."""
    df_limpio = df.copy()
    
    # 1. Columnas críticas (Si faltan, el log es ruido)
    columnas_obligatorias = ['timestamp', 'source_ip', 'dest_ip']
    df_limpio = df_limpio.dropna(subset=columnas_obligatorias)
    
    # 2. Relleno de datos faltantes en columnas secundarias
    if 'user_agent' in df_limpio.columns:
        df_limpio['user_agent'] = df_limpio['user_agent'].fillna('Desconocido')
        
    if 'request_path' in df_limpio.columns:
        df_limpio['request_path'] = df_limpio['request_path'].fillna('/')
        
    return df_limpio

# Ejecucion de limpieza
df_sin_nulos = limpiar_nulos(df_crudo)
print(f"Filas retenidas después de limpiar nulos: {df_sin_nulos.shape[0]:,}")

NameError: name 'df_crudo' is not defined

## 3.5 Validación de Integridad de Direcciones IP
Los sensores de red ocasionalmente pueden generar logs corruptos donde los campos de origen y destino no contienen direcciones IP reales, sino fragmentos de texto o errores de codificación. Para evitar que esta "basura" contamine el *Feature Engineering* posterior, aplicamos un filtro de expresiones regulares (Regex) que descartará cualquier registro cuyas IPs no cumplan con la estructura estándar de una IPv4 (X.X.X.X).

In [ ]:
def filtrar_ips_invalidas(df):
    """
    Filtra y descarta los registros donde source_ip o dest_ip no tengan 
    un formato de dirección IPv4 válido.
    """
    df_limpio = df.copy()
    
    # Expresión regular para validar formato IPv4 (1 a 3 dígitos separados por puntos)
    regex_ipv4 = r'^(?:\d{1,3}\.){3}\d{1,3}$'
    
    # Verificamos qué filas cumplen con el formato exacto en ambas columnas
    mascara_source = df_limpio['source_ip'].astype(str).str.match(regex_ipv4)
    mascara_dest = df_limpio['dest_ip'].astype(str).str.match(regex_ipv4)
    
    # Filtramos manteniendo solo las filas correctas (el operador & significa "Y")
    df_filtrado = df_limpio[mascara_source & mascara_dest]
    
    # Calculamos cuánta "basura" eliminamos para llevar el registro
    borradas = len(df_limpio) - len(df_filtrado)
    if borradas > 0:
        print(f"Se descartaron {borradas:,} filas por contener IPs inválidas o corruptas.")
    else:
        print("Validación completada: Todas las IPs tienen un formato válido.")
        
    return df_filtrado

# Ejecutamos este filtro pasándole el dataset que ya no tiene nulos (Paso 3)
df_ips_validas = filtrar_ips_invalidas(df_sin_nulos)

print(f"Filas retenidas después de validar IPs: {df_ips_validas.shape[0]:,}")

## 4. Transformación de Variables (Casting)
Los modelos de datos necesitan interpretar las fechas como tiempo real y no solo leerlas como cadenas de texto. Por esta razón, convertimos la información de la columna timestamp al formato de fechas propio de Pandas (datetime).

In [4]:
def transformar_fechas(df):
    """Convierte la columna de texto timestamp a un formato de tiempo."""
    df_transformado = df.copy()
    
    # errors='coerce' convierte cualquier texto corrupto en un valor nulo (NaT)
    df_transformado['timestamp'] = pd.to_datetime(df_transformado['timestamp'], errors='coerce')
    
    # Eliminar las filas que hayan quedado con fechas inválidas
    df_transformado = df_transformado.dropna(subset=['timestamp'])
    
    return df_transformado

# Ejecutar el casting
df_con_fechas = transformar_fechas(df_ips_validas)
print("Tipos de datos actualizados:")
print(df_con_fechas[['timestamp', 'source_ip']].dtypes)

Tipos de datos actualizados:
timestamp    datetime64[us]
source_ip               str
dtype: object


## 5. Normalización de Datos
Para preparar el dataset para un futuro modelado algorítmico, aplicamos un escalamiento **Min-Max** a la variable `bytes_transferred`. Esto comprime todos los valores de transferencia de red en un rango entre 0 y 1, evitando que flujos inusualmente pesados sesguen los modelos de detección.

In [5]:
def escalar_bytes(df):
    """Escala los bytes transferidos a un valor entre 0 y 1 para facilitar el modelado."""
    df_escalado = df.copy()
    
    if 'bytes_transferred' in df_escalado.columns:
        minimo = df_escalado['bytes_transferred'].min()
        maximo = df_escalado['bytes_transferred'].max()
        
        # Fórmula matemática Min-Max
        if maximo > minimo:
            df_escalado['bytes_escalados'] = (df_escalado['bytes_transferred'] - minimo) / (maximo - minimo)
        else:
            df_escalado['bytes_escalados'] = 0.0
            
    return df_escalado

# Ejecutar la normalización
df_escalado = escalar_bytes(df_con_fechas)
print("Escalamiento completado. Muestra de los datos transformados:")
display(df_escalado[['timestamp', 'bytes_transferred', 'bytes_escalados']].head())

Escalamiento completado. Muestra de los datos transformados:


,timestamp,bytes_transferred,bytes_escalados
0,2024-05-01,10889,0.216212
1,2024-07-18,36522,0.729900
2,2024-04-07,20652,0.411864
3,2024-10-26,5350,0.105210
4,2024-10-31,40691,0.813447


## 6. Transformación de Texto a Números (Codificación)
Para poder aplicar cálculos matemáticos a nuestros datos, es necesario que toda la información sea numérica, ya que las fórmulas no pueden procesar palabras. Para estandarizar los logs, transformaremos el texto aplicando dos técnicas:

1. **Variables sin orden específico (`action`, `log_type`):** Aplicamos la técnica de *One-Hot Encoding*. Como no podemos decir que un equipo "firewall" vale más matemáticamente que un "ids", separamos estas categorías en columnas nuevas que solo contienen 0 y 1. Esto evita que el sistema asuma por error que un dispositivo es más importante que el otro.
2. **Variable de nivel de amenaza (`threat_label`):** Aplicamos la técnica de *Label Encoding*. A diferencia del caso anterior, las etiquetas de amenaza sí tienen un orden lógico de riesgo. Por lo tanto, reemplazamos el texto directamente por números que reflejan su nivel de severidad (`benign` = 0, `suspicious` = 1, `malicious` = 2).

In [6]:
def codificar_categoricas(df):
    """
    Transforma las variables de texto en valores numéricos interpretables por los calculos matematicos.
    """
    df_encoded = df.copy()
    
    # 1. One-Hot Encoding para características
    columnas_nominales = ['action', 'log_type']
    df_encoded = pd.get_dummies(df_encoded, columns=columnas_nominales, dtype=int)
    
    # 2. Label Encoding para la variable de threat_label
    if 'threat_label' in df_encoded.columns:
        mapa_amenazas = {'benign': 0, 'suspicious': 1, 'malicious': 2}
        df_encoded['threat_label_encoded'] = df_encoded['threat_label'].map(mapa_amenazas)
        df_encoded = df_encoded.drop(columns=['threat_label'])
        
    return df_encoded

# Aquí tomamos el df_escalado del paso anterior y creamos el df_final numérico
df_final = codificar_categoricas(df_escalado)

print("Codificación completada. Muestra de las nuevas columnas numéricas:")
display(df_final.head())

Codificación completada. Muestra de las nuevas columnas numéricas:


,timestamp,source_ip,dest_ip,protocol,bytes_transferred,user_agent,request_path,bytes_escalados,action_allowed,action_blocked,log_type_application,log_type_firewall,log_type_ids,threat_label_encoded
0,2024-05-01,192.168.1.125,192.168.1.124,TCP,10889,Nmap Scripting Engine,/,0.216212,0,1,0,1,0,0
1,2024-07-18,192.168.1.201,192.168.1.201,ICMP,36522,Nmap Scripting Engine,/,0.729900,0,1,1,0,0,0
2,2024-04-07,192.168.1.248,192.168.1.15,HTTP,20652,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,/login,0.411864,1,0,1,0,0,0
3,2024-10-26,192.168.1.236,192.168.1.219,HTTP,5350,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,/login,0.105210,1,0,1,0,0,0
4,2024-10-31,192.168.1.221,192.168.1.61,ICMP,40691,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,/,0.813447,1,0,1,0,0,0


## 7. Validación Técnica del Código
El último paso consiste en verificar que todas las transformaciones aplicadas a lo largo de este *notebook* resulten en un *dataset* consistente, sin nulos residuales y con los tipos de datos correctos, asegurando su trazabilidad para la Fase 3.

In [7]:
# Comprobación de resultados intermedios y excepciones
print("--- RESUMEN DE VALIDACIÓN TÉCNICA ---")

# 1. Comprobación de Nulos
nulos_restantes = df_final.isnull().sum().sum()
print(f"[Verificación] Valores nulos restantes en el dataset completo: {nulos_restantes}")

# 2. Comprobación de Tipos
tipo_fecha = df_final['timestamp'].dtype
print(f"[Verificación] Tipo de dato de 'timestamp': {tipo_fecha}")

# 3. Comprobación de Escalamiento
min_esc = df_final['bytes_escalados'].min()
max_esc = df_final['bytes_escalados'].max()
print(f"[Verificación] Rango de escalamiento: Mínimo {min_esc:.1f} / Máximo {max_esc:.1f}")

if nulos_restantes == 0 and min_esc >= 0.0 and max_esc <= 1.0:
    print("\nVALIDACIÓN EXITOSA: El dataset es íntegro, estable y apto para exportación.")
else:
    print("\nADVERTENCIA: Existen inconsistencias en el preprocesamiento.")

--- RESUMEN DE VALIDACIÓN TÉCNICA ---
[Verificación] Valores nulos restantes en el dataset completo: 0
[Verificación] Tipo de dato de 'timestamp': datetime64[us]
[Verificación] Rango de escalamiento: Mínimo 0.0 / Máximo 1.0

VALIDACIÓN EXITOSA: El dataset es íntegro, estable y apto para exportación.


## 8. Exportación del Dataset Procesado
Para garantizar la eficiencia de las fases posteriores, el dataset limpio, casteado y normalizado se exporta al directorio `data/processed/`. Esto evitará tener que repetir el costo computacional de esta fase durante la Fase 3.

In [8]:
# Definimos la ruta de salida
ruta_salida = '../data/processed/cybersecurity_logs_procesados.csv'

print("Filtrando columnas redundantes antes de exportar...")
# Se elimina la columna original porque ya tenemos la escalada
columnas_a_eliminar = ['bytes_transferred'] 

df_exportar = df_final.drop(columns=columnas_a_eliminar)

print("Iniciando exportación del dataset procesado...")
df_exportar.to_csv(ruta_salida, index=False)

print(f"¡Exportación exitosa! El dataset está listo en: {ruta_salida}")

Filtrando columnas redundantes antes de exportar...
Iniciando exportación del dataset procesado...
¡Exportación exitosa! El dataset está listo en: ../data/processed/cybersecurity_logs_procesados.csv
